# QUANT CLUB Freshers Selection Task - 1
## Algorithmic Trading System for RELIANCE.NS

**Objective:** Build a binary classification model to predict whether Reliance Industries' closing price will be higher 5 trading days from now, and simulate a long/short trading strategy.

- **Ticker:** RELIANCE.NS
- **Period:** January 1, 2011 to January 1, 2024
- **Train Set:** Jan 2011 – Dec 2020 (10 years)
- **Test Set:** Jan 2021 – Jan 2024 (3 years)
- **Target:** 1 (UP) if Close(T+5) > Close(T), else 0 (DOWN)

In [ ]:
# Install required libraries
!pip install yfinance scikit-learn xgboost pandas numpy matplotlib seaborn ta -q

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import (f1_score, confusion_matrix, classification_report,
                             ConfusionMatrixDisplay, accuracy_score, precision_score, recall_score)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

import ta

plt.style.use('seaborn-v0_8-darkgrid')
pd.set_option('display.max_columns', None)
print('All libraries imported successfully.')

## 1. Data Extraction
Fetching 13 years of daily OHLCV data for RELIANCE.NS using `yfinance`.

In [ ]:
# Fetch RELIANCE.NS data
ticker = 'RELIANCE.NS'
df = yf.download(ticker, start='2011-01-01', end='2024-01-02', auto_adjust=False)

# Flatten multi-level columns if present
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

# Keep relevant columns and sort
df = df[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
df.sort_index(inplace=True)
df.dropna(inplace=True)

print(f'Data shape: {df.shape}')
print(f'Date range: {df.index.min().date()} to {df.index.max().date()}')
df.head()

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [3, 1]})

axes[0].plot(df.index, df['Close'], color='#1f77b4', linewidth=0.8)
axes[0].set_title('RELIANCE.NS - Daily Close Price (2011-2024)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Close Price (INR)')
axes[0].axvline(pd.Timestamp('2021-01-01'), color='red', linestyle='--', label='Train/Test Split')
axes[0].legend()

axes[1].bar(df.index, df['Volume'], color='#2ca02c', alpha=0.6, width=2)
axes[1].set_title('Trading Volume', fontsize=12)
axes[1].set_ylabel('Volume')

plt.tight_layout()
plt.show()

print('\nDescriptive Statistics:')
df.describe()

## 3. Target Variable Creation

**Target:** 1 (UP) if `Close(T+5) > Close(T)`, else 0 (DOWN).

We shift Close price by -5 to get the future close, ensuring no look-ahead bias in feature calculation.

In [ ]:
# Create target: 1 if Close 5 days ahead > today's Close, else 0
df['Close_T5'] = df['Close'].shift(-5)
df['Target'] = (df['Close_T5'] > df['Close']).astype(int)

# Also store Open_T1 and Close_T5 for trading simulation later
df['Open_T1'] = df['Open'].shift(-1)

print('Target Distribution (before dropping NaNs):')
print(df['Target'].value_counts())
print(f'\nUP ratio: {df["Target"].mean():.4f}')

## 4. Feature Engineering (Alpha Generation)

We engineer features using **only data available up to Day T** to prevent look-ahead bias.

### Feature Categories:
1. **Price Transformations** – Log returns, rolling Z-scores
2. **Technical Indicators** – RSI, MACD, Bollinger Bands, ATR, etc.
3. **Volume Features** – Volume ratios, OBV
4. **Momentum & Mean Reversion** – Rate of change, distance from moving averages

In [ ]:
# ---- 4.1 Price Transformations ----
df['Log_Return'] = np.log(df['Close'] / df['Close'].shift(1))
df['Log_Return_5d'] = np.log(df['Close'] / df['Close'].shift(5))
df['Log_Return_10d'] = np.log(df['Close'] / df['Close'].shift(10))
df['Log_Return_20d'] = np.log(df['Close'] / df['Close'].shift(20))

# Rolling Z-score of Close (20-day)
rolling_mean = df['Close'].rolling(20).mean()
rolling_std = df['Close'].rolling(20).std()
df['ZScore_20'] = (df['Close'] - rolling_mean) / rolling_std

# Rolling Z-score of Close (50-day)
rolling_mean_50 = df['Close'].rolling(50).mean()
rolling_std_50 = df['Close'].rolling(50).std()
df['ZScore_50'] = (df['Close'] - rolling_mean_50) / rolling_std_50

# Realized Volatility
df['Volatility_20'] = df['Log_Return'].rolling(20).std() * np.sqrt(252)
df['Volatility_60'] = df['Log_Return'].rolling(60).std() * np.sqrt(252)

print('Price transformation features created.')

In [ ]:
# ---- 4.2 Technical Indicators (using `ta` library) ----

# RSI (14-day)
df['RSI_14'] = ta.momentum.RSIIndicator(df['Close'], window=14).rsi()

# MACD
macd = ta.trend.MACD(df['Close'])
df['MACD'] = macd.macd()
df['MACD_Signal'] = macd.macd_signal()
df['MACD_Hist'] = macd.macd_diff()

# Bollinger Bands
bb = ta.volatility.BollingerBands(df['Close'], window=20, window_dev=2)
df['BB_High'] = bb.bollinger_hband()
df['BB_Low'] = bb.bollinger_lband()
df['BB_Width'] = (df['BB_High'] - df['BB_Low']) / df['Close']
df['BB_Pct'] = bb.bollinger_pband()  # %B indicator

# Average True Range (ATR)
df['ATR_14'] = ta.volatility.AverageTrueRange(df['High'], df['Low'], df['Close'], window=14).average_true_range()
df['ATR_Pct'] = df['ATR_14'] / df['Close']  # ATR as percentage of price

# Stochastic Oscillator
stoch = ta.momentum.StochasticOscillator(df['High'], df['Low'], df['Close'])
df['Stoch_K'] = stoch.stoch()
df['Stoch_D'] = stoch.stoch_signal()

# ADX
adx = ta.trend.ADXIndicator(df['High'], df['Low'], df['Close'], window=14)
df['ADX'] = adx.adx()

# CCI
df['CCI'] = ta.trend.CCIIndicator(df['High'], df['Low'], df['Close'], window=20).cci()

print('Technical indicator features created.')

In [ ]:
# ---- 4.3 Volume Features ----
df['Volume_SMA_20'] = df['Volume'].rolling(20).mean()
df['Volume_Ratio'] = df['Volume'] / df['Volume_SMA_20']

# On Balance Volume (OBV)
df['OBV'] = ta.volume.OnBalanceVolumeIndicator(df['Close'], df['Volume']).on_balance_volume()
df['OBV_Pct'] = df['OBV'].pct_change(periods=5)

# Money Flow Index
df['MFI'] = ta.volume.MoneFlowIndexIndicator(df['High'], df['Low'], df['Close'], df['Volume'], window=14).money_flow_index()

print('Volume features created.')

In [ ]:
# ---- 4.4 Momentum & Mean Reversion ----
# Moving Average distances
for w in [10, 20, 50, 200]:
    sma = df['Close'].rolling(w).mean()
    df[f'Dist_SMA_{w}'] = (df['Close'] - sma) / sma

# EMA distances
for w in [12, 26]:
    ema = df['Close'].ewm(span=w, adjust=False).mean()
    df[f'Dist_EMA_{w}'] = (df['Close'] - ema) / ema

# Rate of Change
df['ROC_5'] = df['Close'].pct_change(5)
df['ROC_10'] = df['Close'].pct_change(10)
df['ROC_20'] = df['Close'].pct_change(20)

# Candle body ratio
df['Body_Ratio'] = (df['Close'] - df['Open']) / (df['High'] - df['Low'] + 1e-8)

# High-Low range as pct of close
df['HL_Pct'] = (df['High'] - df['Low']) / df['Close']

print('Momentum & Mean Reversion features created.')
print(f'\nTotal columns after feature engineering: {df.shape[1]}')

## 5. Data Cleaning & Chronological Train/Test Split

- Drop rows with NaN values (from rolling calculations and target shift)
- **No shuffling** – strict chronological split
- Train: Jan 2011 – Dec 2020 | Test: Jan 2021 – Jan 2024

In [ ]:
# Define feature columns (exclude target, helper cols, raw OHLCV prices)
exclude_cols = ['Open', 'High', 'Low', 'Close', 'Volume', 'Target',
                'Close_T5', 'Open_T1', 'BB_High', 'BB_Low', 'OBV',
                'Volume_SMA_20']

feature_cols = [c for c in df.columns if c not in exclude_cols]
print(f'Feature columns ({len(feature_cols)}):')
print(feature_cols)

# Drop NaN rows
df_clean = df.dropna(subset=feature_cols + ['Target', 'Open_T1', 'Close_T5']).copy()
print(f'\nClean data shape: {df_clean.shape}')
print(f'Date range: {df_clean.index.min().date()} to {df_clean.index.max().date()}')

In [ ]:
# Chronological split
train_end = '2020-12-31'
test_start = '2021-01-01'

train_df = df_clean[df_clean.index <= train_end].copy()
test_df = df_clean[df_clean.index >= test_start].copy()

X_train = train_df[feature_cols]
y_train = train_df['Target']
X_test = test_df[feature_cols]
y_test = test_df['Target']

print(f'Train: {X_train.shape[0]} samples ({train_df.index.min().date()} to {train_df.index.max().date()})')
print(f'Test:  {X_test.shape[0]} samples ({test_df.index.min().date()} to {test_df.index.max().date()})')
print(f'\nTrain Target Distribution:\n{y_train.value_counts(normalize=True)}')
print(f'\nTest Target Distribution:\n{y_test.value_counts(normalize=True)}')

## 6. Feature Scaling

**Critical:** We fit the `StandardScaler` **only on training data** to prevent data leakage.

In [ ]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_cols, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=feature_cols, index=X_test.index)

print('Scaling complete. Scaler fitted on training data only.')
print(f'Train scaled mean (should be ~0): {X_train_scaled.mean().mean():.6f}')
print(f'Train scaled std  (should be ~1): {X_train_scaled.std().mean():.6f}')

## 7. Model Selection & Hyperparameter Tuning

We compare multiple models with `TimeSeriesSplit` cross-validation and `GridSearchCV`.

**Models:**
1. Logistic Regression (baseline)
2. Random Forest
3. XGBoost
4. Gradient Boosting

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

models = {
    'LogisticRegression': {
        'model': LogisticRegression(max_iter=1000, random_state=42),
        'params': {
            'C': [0.01, 0.1, 1, 10],
            'penalty': ['l2'],
            'solver': ['lbfgs']
        }
    },
    'RandomForest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'n_estimators': [100, 200, 300],
            'max_depth': [5, 10, 15, None],
            'min_samples_split': [5, 10, 20],
            'min_samples_leaf': [2, 5, 10]
        }
    },
    'XGBoost': {
        'model': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
        'params': {
            'n_estimators': [100, 200, 300],
            'max_depth': [3, 5, 7],
            'learning_rate': [0.01, 0.05, 0.1],
            'subsample': [0.7, 0.8, 1.0],
            'colsample_bytree': [0.7, 0.8, 1.0]
        }
    },
    'GradientBoosting': {
        'model': GradientBoostingClassifier(random_state=42),
        'params': {
            'n_estimators': [100, 200],
            'max_depth': [3, 5, 7],
            'learning_rate': [0.01, 0.05, 0.1],
            'subsample': [0.7, 0.8, 1.0]
        }
    }
}

results = {}
best_models = {}

for name, config in models.items():
    print(f'\n{"="*50}')
    print(f'Training: {name}')
    print(f'{"="*50}')
    
    grid = GridSearchCV(
        config['model'],
        config['params'],
        cv=tscv,
        scoring='f1',
        n_jobs=-1,
        verbose=0
    )
    grid.fit(X_train_scaled, y_train)
    
    y_pred = grid.predict(X_test_scaled)
    f1 = f1_score(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)
    
    results[name] = {'f1': f1, 'accuracy': acc, 'best_params': grid.best_params_,
                     'cv_score': grid.best_score_}
    best_models[name] = grid.best_estimator_
    
    print(f'Best CV F1: {grid.best_score_:.4f}')
    print(f'Test F1: {f1:.4f} | Test Acc: {acc:.4f}')
    print(f'Best Params: {grid.best_params_}')

# Summary table
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('f1', ascending=False)
print('\n' + '='*60)
print('MODEL COMPARISON (sorted by Test F1):')
print('='*60)
print(results_df[['f1', 'accuracy', 'cv_score']].to_string())

In [ ]:
# Select the best model
best_model_name = results_df.index[0]
best_model = best_models[best_model_name]
print(f'\nBest Model: {best_model_name}')
print(f'Best Params: {results[best_model_name]["best_params"]}')

# Final predictions
y_pred_final = best_model.predict(X_test_scaled)
y_proba_final = best_model.predict_proba(X_test_scaled)[:, 1] if hasattr(best_model, 'predict_proba') else None

## 8. Model Evaluation Metrics

In [ ]:
# Classification Report
print('Classification Report (Test Set):')
print('='*50)
print(classification_report(y_test, y_pred_final, target_names=['DOWN (0)', 'UP (1)']))

# F1 Score
print(f'\nF1 Score: {f1_score(y_test, y_pred_final):.4f}')
print(f'Accuracy: {accuracy_score(y_test, y_pred_final):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_final):.4f}')
print(f'Recall: {recall_score(y_test, y_pred_final):.4f}')

In [ ]:
# Confusion Matrix
fig, ax = plt.subplots(figsize=(7, 5))
cm = confusion_matrix(y_test, y_pred_final)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['DOWN (0)', 'UP (1)'])
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title(f'Confusion Matrix – {best_model_name}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nTrue Negatives: {cm[0,0]}  |  False Positives: {cm[0,1]}')
print(f'False Negatives: {cm[1,0]}  |  True Positives: {cm[1,1]}')

In [ ]:
# Feature Importance
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
elif hasattr(best_model, 'coef_'):
    importances = np.abs(best_model.coef_[0])
else:
    importances = None

if importances is not None:
    feat_imp = pd.Series(importances, index=feature_cols).sort_values(ascending=True)
    
    fig, ax = plt.subplots(figsize=(10, 10))
    feat_imp.tail(20).plot(kind='barh', ax=ax, color='#2196F3')
    ax.set_title(f'Top 20 Feature Importances – {best_model_name}', fontsize=14, fontweight='bold')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.show()
    
    print('\nTop 10 Features:')
    print(feat_imp.tail(10).to_string())
else:
    print('Feature importances not available for this model type.')

## 9. Trading Simulation (Long & Short)

### Trading Rules:
- **Initial Capital:** ₹1,000,000
- **Signal 1 (UP):** Go Long → Buy at Open(T+1), sell at Close(T+5)
- **Signal 0 (DOWN):** Go Short → Short at Open(T+1), cover at Close(T+5)
- **Holding Period:** 5 trading days
- **100% capital allocation per trade**
- **Zero transaction costs**

### Return Formulas:
- Long Return = Close(T+5) / Open(T+1) - 1
- Short Return = Open(T+1) / Close(T+5) - 1

In [ ]:
# Trading simulation on TEST SET only
initial_capital = 1_000_000

# Prepare simulation dataframe
sim_df = test_df[['Open', 'Close', 'Open_T1', 'Close_T5', 'Target']].copy()
sim_df['Prediction'] = y_pred_final

# Drop rows where we don't have Open_T1 or Close_T5
sim_df = sim_df.dropna(subset=['Open_T1', 'Close_T5'])

# Simulate trades
capital = initial_capital
trade_log = []
i = 0

while i < len(sim_df):
    row = sim_df.iloc[i]
    signal = row['Prediction']
    entry_price = row['Open_T1']
    exit_price = row['Close_T5']
    entry_date = sim_df.index[i]
    
    if signal == 1:  # Long
        trade_return = (exit_price / entry_price) - 1
        direction = 'LONG'
    else:  # Short
        trade_return = (entry_price / exit_price) - 1
        direction = 'SHORT'
    
    new_capital = capital * (1 + trade_return)
    
    trade_log.append({
        'Date': entry_date,
        'Direction': direction,
        'Entry_Price': entry_price,
        'Exit_Price': exit_price,
        'Return': trade_return,
        'Capital_Before': capital,
        'Capital_After': new_capital,
        'Actual_Direction': 'UP' if row['Target'] == 1 else 'DOWN'
    })
    
    capital = new_capital
    i += 6  # Hold for 5 days, then wait for next signal (skip 5+1 days)

trades_df = pd.DataFrame(trade_log)
print(f'Total trades executed: {len(trades_df)}')
print(f'Long trades: {(trades_df["Direction"] == "LONG").sum()}')
print(f'Short trades: {(trades_df["Direction"] == "SHORT").sum()}')
trades_df.head(10)

## 10. Financial Performance Metrics

In [ ]:
# Total Return
final_capital = trades_df['Capital_After'].iloc[-1]
total_return = (final_capital / initial_capital - 1) * 100

# Sharpe Ratio (annualized)
risk_free_rate = 0.05  # 5% annual
trade_returns = trades_df['Return'].values
avg_holding = 5  # days
trades_per_year = 252 / avg_holding  # approx 50 trades per year

mean_return = np.mean(trade_returns)
std_return = np.std(trade_returns, ddof=1)
rf_per_trade = risk_free_rate / trades_per_year

sharpe_ratio = (mean_return - rf_per_trade) / std_return * np.sqrt(trades_per_year) if std_return > 0 else 0

print('='*60)
print('FINANCIAL PERFORMANCE SUMMARY (Test Set: 2021-2024)')
print('='*60)
print(f'Initial Capital:    ₹{initial_capital:>15,.2f}')
print(f'Final Capital:      ₹{final_capital:>15,.2f}')
print(f'Total Return:        {total_return:>14.2f}%')
print(f'Annualized Sharpe:   {sharpe_ratio:>14.4f}')
print(f'\nPer-Trade Stats:')
print(f'  Mean Return:       {mean_return*100:>14.4f}%')
print(f'  Std Return:        {std_return*100:>14.4f}%')
print(f'  Win Rate:          {(trade_returns > 0).mean()*100:>14.2f}%')
print(f'  Max Win:           {trade_returns.max()*100:>14.4f}%')
print(f'  Max Loss:          {trade_returns.min()*100:>14.4f}%')
print(f'  Total Trades:      {len(trade_returns):>14d}')

In [ ]:
# Equity Curve
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Plot 1: Equity Curve
equity_curve = [initial_capital] + list(trades_df['Capital_After'])
axes[0].plot(range(len(equity_curve)), equity_curve, 'b-', linewidth=1.5, label='Strategy')
axes[0].axhline(y=initial_capital, color='red', linestyle='--', alpha=0.5, label='Initial Capital')
axes[0].set_title('Equity Curve (Test Period: 2021-2024)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Portfolio Value (INR)')
axes[0].set_xlabel('Trade Number')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Per-trade returns
colors = ['green' if r > 0 else 'red' for r in trades_df['Return']]
axes[1].bar(range(len(trades_df)), trades_df['Return'] * 100, color=colors, alpha=0.7)
axes[1].set_title('Per-Trade Returns (%)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Return (%)')
axes[1].set_xlabel('Trade Number')
axes[1].axhline(y=0, color='black', linewidth=0.5)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Return Distribution
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(trades_df['Return'] * 100, bins=30, color='#2196F3', edgecolor='white', alpha=0.8)
ax.axvline(x=0, color='red', linestyle='--', linewidth=1.5)
ax.axvline(x=trades_df['Return'].mean()*100, color='green', linestyle='--', linewidth=1.5, label=f'Mean: {trades_df["Return"].mean()*100:.2f}%')
ax.set_title('Distribution of Trade Returns', fontsize=14, fontweight='bold')
ax.set_xlabel('Return (%)')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.show()

## 11. Feature Improvement Demonstration

We compare our full-featured model against a baseline with only raw returns to demonstrate the value of feature engineering.

In [ ]:
# Baseline: Only raw log returns as features
baseline_features = ['Log_Return', 'Log_Return_5d', 'Log_Return_10d', 'Log_Return_20d']

scaler_base = StandardScaler()
X_train_base = pd.DataFrame(scaler_base.fit_transform(train_df[baseline_features]),
                            columns=baseline_features, index=train_df.index)
X_test_base = pd.DataFrame(scaler_base.transform(test_df[baseline_features]),
                           columns=baseline_features, index=test_df.index)

# Train same model type with default params
if best_model_name == 'XGBoost':
    base_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
elif best_model_name == 'RandomForest':
    base_model = RandomForestClassifier(random_state=42)
elif best_model_name == 'GradientBoosting':
    base_model = GradientBoostingClassifier(random_state=42)
else:
    base_model = LogisticRegression(max_iter=1000, random_state=42)

base_model.fit(X_train_base, y_train)
y_pred_base = base_model.predict(X_test_base)

print('FEATURE ENGINEERING IMPACT')
print('='*50)
print(f'{"Metric":<25} {"Baseline":>12} {"Full Model":>12} {"Improvement":>12}')
print('-'*61)
f1_base = f1_score(y_test, y_pred_base)
f1_full = f1_score(y_test, y_pred_final)
acc_base = accuracy_score(y_test, y_pred_base)
acc_full = accuracy_score(y_test, y_pred_final)
print(f'{"F1 Score":<25} {f1_base:>12.4f} {f1_full:>12.4f} {f1_full-f1_base:>+12.4f}')
print(f'{"Accuracy":<25} {acc_base:>12.4f} {acc_full:>12.4f} {acc_full-acc_base:>+12.4f}')

## 12. Research Narrative & Methodology

### Methodology

**Why this pipeline?**
- We chose a multi-model comparison approach (Logistic Regression, Random Forest, XGBoost, Gradient Boosting) because financial time-series prediction is inherently noisy, and no single model dominates across all regimes.
- `TimeSeriesSplit` was used instead of random CV to respect the temporal ordering and prevent future data from leaking into training folds.
- `GridSearchCV` with F1 scoring ensures models are optimized for balanced classification, not just accuracy.

**Feature Engineering Rationale:**
- **Log returns** make prices stationary and comparable across time periods.
- **Rolling Z-scores** normalize price levels relative to recent history, capturing mean-reversion signals.
- **RSI, MACD, Bollinger Bands** capture momentum, trend, and volatility regimes — established technical analysis concepts.
- **Volume features** (Volume Ratio, MFI, OBV) capture changes in market participation and smart money flow.
- **ATR as % of price** normalizes volatility across the price range of the stock over 13 years.
- **SMA/EMA distance** features capture trend positioning at multiple timeframes.

### Experimentation & Failures

Features/approaches that were considered but may not add value:
- **Raw OHLCV prices** were excluded as features because they are non-stationary and would cause the model to memorize price levels rather than learn patterns.
- **Macroeconomic data** (e.g., India VIX, crude oil prices, USD/INR exchange rates) could potentially improve the model but were not included in this iteration to keep the pipeline focused on technical features.
- **Sentiment signals** from news/social media could add value but require additional data sources and NLP pipelines.

### Key Observations
- Financial markets are inherently noisy; F1 scores around 0.50-0.60 are realistic and should not be dismissed.
- The 5-day horizon is a reasonable balance between noise (1-day) and reduced sample size (longer horizons).
- The long/short strategy capitalizes on directional predictions in both market regimes.

### Potential Improvements
1. Ensemble methods: Stacking multiple models
2. Feature selection: Using SHAP values or recursive feature elimination
3. Regime detection: Separate models for trending vs. ranging markets
4. Alternative targets: Probability-weighted position sizing instead of binary signals